# Indexing voi mo hinh xac suat

In [81]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer, RegexTokenizer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

#import pytrec_eval
import math

**1. Tien xu ly du lieu**

In [82]:
import os, py_vncorenlp

In [83]:
def load_vncorenlp(model_dir):
    
    jar_path = os.path.join(model_dir, "VnCoreNLP-1.2.jar")
    if not os.path.exists(jar_path):
        raise FileNotFoundError(f"❌ Missing {jar_path}")
    
    if not os.path.exists(os.path.join(model_dir, "models")):
        raise FileNotFoundError(f"❌ Missing model folder at {model_dir}/models")
    
    return py_vncorenlp.VnCoreNLP(
        annotators=["wseg", "pos", "ner", "parse"],
        save_dir=model_dir,
        max_heap_size='-Xmx2g'
    )


In [84]:
#model = load_vncorenlp(r"C:/Users/mt200/OneDrive/Desktop/AI/InformationRetrieval/Project_InformationRetrieval/Reference/VnCoreNLP-master/")
#print(model.word_segment("tôi yêu xử lý ngôn ngữ tự nhiên."))

In [85]:
#
from pyvi import ViTokenizer

def word_segment(text):
    return ViTokenizer.tokenize(text)

print(word_segment("tôi yêu xử lý ngôn ngữ tự nhiên."))


tôi yêu xử_lý ngôn_ngữ tự_nhiên .


In [86]:
#
#puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
#stoplist = set()
#with open(r"C:\Users\mt200\OneDrive\Desktop\AI\InformationRetrieval\Project_InformationRetrieval\4.Indexing\stopword\vietnamese-stopwords-dash.txt", "r", encoding="utf-8") as f:
#    for line in f:
#        stoplist.add(line.strip())

In [87]:
#
#print("Thư mục hiện tại:", os.getcwd())

In [88]:
#
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
stoplist = set()
with open("../4.Indexing/stopword/vietnamese-stopwords-dash.txt", "r", encoding="utf-8") as f:
    for line in f:
        stoplist.add(line.strip())

In [89]:
def preprocess(tok, punctlist=puncts, stopwords=stoplist):
  tok = tok.lower()
  if tok.isdigit():
    return None
  if tok.isnumeric():
    return None
  if tok in punctlist:
    return None
  if tok in stopwords:
    return None
  return tok

**2. Lap chi muc**

In [ ]:
def indexing(src, idx="ind"):
    if not src.endswith('/'):
        src += '/'

    # Ensure index directory exists
    os.makedirs(idx, exist_ok=True)

    schema = Schema(docid=STORED(), content=TEXT(stored=True, analyzer=StandardAnalyzer()))
    ix = create_in(idx, schema)
    writer = ix.writer()

    files = os.listdir(src)
    for f in files:
        file_path = os.path.join(src, f)
        if not os.path.isfile(file_path):
            continue

        encodings_to_try = ['utf-8', 'utf-16', 'cp1252']
        content = None
        for enc in encodings_to_try:
            try:
                with open(file_path, encoding=enc) as r:
                    content = r.read()
                break
            except UnicodeDecodeError:
                continue

        if content is None:
            print(f"⚠️ Skipped file {f}: could not decode with {encodings_to_try}")
            continue

        try:
            terms = []
            for sent in sent_tokenize(content.strip()):
                #temp_sent_dash = model.word_segment(sent)
                temp_sent_dash = word_segment(sent)

                if isinstance(temp_sent_dash, list):
                    temp_sent_dash = " ".join(temp_sent_dash)

                for tok in word_tokenize(temp_sent_dash):
                    tok = preprocess(tok)
                    if tok:
                        terms.append(tok)

            cont = " ".join(terms)
            writer.add_document(docid=f.split(".")[0], content=cont)

        except Exception as e:
            print(f"⚠️ Skipped file {f}: {e}")

    writer.commit()
    print(f"Indexing completed. Index stored in: {idx}")
    

In [ ]:
#
def indexing_docs(src, idx="ind"):
    if not src.endswith('/'):
        src += '/'

    # Ensure index directory exists
    os.makedirs(idx, exist_ok=True)

    schema = Schema(docid=STORED(), content=TEXT(stored=True, analyzer=RegexTokenizer()))
    ix = create_in(idx, schema)
    writer = ix.writer()

    files = os.listdir(src)
    for f in files:
        file_path = os.path.join(src, f)
        if not os.path.isfile(file_path):
            continue

        encodings_to_try = ['utf-8', 'utf-16', 'cp1252', 'utf-8-sig']
        content = None
        for enc in encodings_to_try:
            try:
                with open(file_path, encoding=enc) as r:
                    content = r.read()
                break
            except UnicodeDecodeError:
                continue

        if content is None:
            print(f"Skipped file {f}: could not decode with {encodings_to_try}")
            continue

        try:
            terms = []

            segmented = word_segment(content)
            
            for tok in segmented.split():
                cleaned = preprocess(tok)  
                if cleaned:
                    terms.append(cleaned)

            cont = " ".join(terms)
            writer.add_document(docid=f.split(".")[0], content=cont)

        except Exception as e:
            print(f"Skipped file {f}: {e}")

    writer.commit()
    print(f"Indexing completed. Index stored in: {idx}")

In [92]:
#indexing("C:/Users/mt200/OneDrive/Desktop/AI/InformationRetrieval/Project_InformationRetrieval/1.CollectingDocuments/data_clean", "ind")

In [93]:
#
#print("Thư mục hiện tại:", os.getcwd())

In [94]:
#
data_dir = "../1.CollectingDocuments/data_clean"
indexing_docs(data_dir, "ind")

✅ Indexing completed. Index stored in: ind


In [95]:
from whoosh.index import open_dir

ix = open_dir("ind")
reader = ix.reader()
field = list(reader.schema._fields.keys())[0]
terms = list(reader.field_terms(field))[:1000]
print(f"1000 terms đầu: {terms}")
reader.close()

1000 terms đầu: ['0000ff', '000ha', '000m', '000m2', '000m3', '000m²', '000vnd', '000vnđ', '000đ', '00m', '035m', '039m', '03b', '03ha', '03m', '05h30', '05m', '068m', '06h30', '07g00', '07h00', '07m', '08vp', '093f69', '0h', '0k', '0m', '1000m', '1000mm', '100k', '100km', '100m', '1040m', '106m3', '10b', '10h', '10h00', '10h15', '10h30', '10h45', '10ha', '10k', '10km', '10m', '10p', '110km', '110km2', '111d', '113m', '114m', '11b', '11h', '11h00', '11h15', '11h20', '11h30', '11ha', '11km', '11m', '12000ha', '120k', '120km', '120m', '122m', '125km', '12a', '12b', '12h', '12h00', '12h30', '12ha', '12km', '12km2', '130km', '133m', '135k', '137m2', '138a', '13a', '13h', '13h00', '13h30', '13h45', '13km', '13m', '13m2', '13th', '1400ha', '1400m', '140ha', '143m', '144b', '145m', '1487m', '14h', '14h00', '14h20', '14h30', '14h45', '14km', '14m', '1500m', '1500m2', '150c', '150ha', '150k', '150km', '150m', '151ha', '155a', '15a', '15h', '15h00', '15h15', '15h30', '15h45', '15ha', '15km', '15

**3. Truy van**

In [ ]:
from whoosh.qparser import QueryParser
from whoosh.index import open_dir
from whoosh import scoring  

def search_query(idx="ind", query_str=""):
    """
    Thực hiện truy vấn trên chỉ mục Whoosh bằng mô hình xác suất BM25.
    - idx: đường dẫn đến thư mục chỉ mục
    - query_str: chuỗi truy vấn (từ khóa người dùng)
    """
    try:
        # Mở thư mục chỉ mục
        ix = open_dir(idx)

        # Tạo parser cho trường 'content'
        qp = QueryParser("content", schema=ix.schema)
        q = qp.parse(query_str)

        # ⚡ Sử dụng mô hình BM25 thay vì TF-IDF
        with ix.searcher(weighting=scoring.BM25F()) as searcher:
            
            results = searcher.search(q, limit=10)

            if not results:
                print("Không tìm thấy tài liệu phù hợp.")
                return

            print(f"Tìm thấy {len(results)} tài liệu liên quan (BM25):\n")
            for rank, hit in enumerate(results, start=1):
                print(f"{rank}. DocID: {hit['docid']}")
                snippet = hit['content'][:150].replace('\n', ' ')
                print(f"   Trích đoạn: {snippet}...\n")

    except Exception as e:
        print(f"Lỗi khi truy vấn: {e}")

In [97]:
#search_query("ind", "biển đảo Việt Nam")

2POISSON

In [135]:
from whoosh.index import open_dir
import numpy as np

def collect_term_statistics(index_dir):
    ix = open_dir(index_dir)
    reader = ix.reader()
    term_stats = {}

    fieldname = list(reader.schema._fields.keys())[0]
    N = reader.doc_count()

    for term in reader.lexicon(fieldname):
        tf_list = np.zeros(N, dtype=int)

        for docnum, freq in reader.postings(fieldname, term).items_as("frequency"):
            tf_list[docnum] = freq

        term_stats[term] = tf_list

    return term_stats


In [136]:
import numpy as np
from scipy.stats import poisson

def em_two_poisson(tf_values, max_iter=100, tol=1e-6):
    """
    EM algorithm để ước lượng λ, μ1, μ2
    
    Input:
        tf_values: list TF values của 1 term [0, 1, 0, 3, 1, ...]
    Output:
        λ (lambda_mix): xác suất term thuộc elite
        μ1 (mu_elite): trung bình Poisson của elite
        μ2 (mu_non_elite): trung bình Poisson của non-elite
    """
    tf_values = np.array(tf_values)
    n = len(tf_values)
    
    # Initialize parameters
    lambda_mix = 0.5
    mu_elite = np.mean(tf_values) * 1.5 
    mu_non_elite = np.mean(tf_values) * 0.5  
    
    for iteration in range(max_iter):
        # Tính posterior probability: P(elite | TF)
        p_elite = lambda_mix * poisson.pmf(tf_values, mu_elite)
        p_non_elite = (1 - lambda_mix) * poisson.pmf(tf_values, mu_non_elite)
        
        # Tránh chia cho 0
        total = p_elite + p_non_elite + 1e-6
        gamma = p_elite / total 
        
        # Update parameters
        lambda_new = np.mean(gamma)
        mu_elite_new = np.sum(gamma * tf_values) / (np.sum(gamma) + 1e-6)
        mu_non_elite_new = np.sum((1 - gamma) * tf_values) / (np.sum(1 - gamma) + 1e-6)
        
        # Check convergence
        if (abs(lambda_new - lambda_mix) < tol and 
            abs(mu_elite_new - mu_elite) < tol and 
            abs(mu_non_elite_new - mu_non_elite) < tol):
            break
        
        lambda_mix = lambda_new
        mu_elite = mu_elite_new
        mu_non_elite = mu_non_elite_new
    
    return {
        'lambda': lambda_mix,
        'mu_elite': mu_elite,
        'mu_non_elite': mu_non_elite
    }

In [137]:
def train_two_poisson_model(index_dir, output_file='two_poisson_params.pkl'):
    """
    Train Two-Poisson cho tất cả terms và lưu tham số
    """
    import pickle
    
    # Bước 1: Thu thập statistics
    print("Collecting term statistics...")
    term_stats = collect_term_statistics(index_dir)
    
    # Bước 2: Train EM cho từng term
    print("Training Two-Poisson model...")
    term_params = {}
    
    for i, (term, tf_list) in enumerate(term_stats.items()):
        if len(tf_list) < 10:  
            continue
        
        params = em_two_poisson(tf_list)
        term_params[term] = params
        
        if i % 100 == 0:
            print(f"Processed {i}/{len(term_stats)} terms")
    
    # Bước 3: Lưu tham số
    with open(output_file, 'wb') as f:
        pickle.dump(term_params, f)
    
    print(f"Saved parameters to {output_file}")
    return term_params

In [138]:
params = train_two_poisson_model(index_dir="ind")

Training Two-Poisson model...
Processed 0/19390 terms
Processed 100/19390 terms
Processed 200/19390 terms
Processed 300/19390 terms
Processed 400/19390 terms
Processed 500/19390 terms
Processed 600/19390 terms
Processed 700/19390 terms
Processed 800/19390 terms
Processed 900/19390 terms
Processed 1000/19390 terms
Processed 1100/19390 terms
Processed 1200/19390 terms
Processed 1300/19390 terms
Processed 1400/19390 terms
Processed 1500/19390 terms
Processed 1600/19390 terms
Processed 1700/19390 terms
Processed 1800/19390 terms
Processed 1900/19390 terms
Processed 2000/19390 terms
Processed 2100/19390 terms
Processed 2200/19390 terms
Processed 2300/19390 terms
Processed 2400/19390 terms
Processed 2500/19390 terms
Processed 2600/19390 terms
Processed 2700/19390 terms
Processed 2800/19390 terms
Processed 2900/19390 terms
Processed 3000/19390 terms
Processed 3100/19390 terms
Processed 3200/19390 terms
Processed 3300/19390 terms
Processed 3400/19390 terms
Processed 3500/19390 terms
Processed 

In [139]:
from whoosh.scoring import WeightingModel, BaseScorer
import pickle
from scipy.stats import poisson
import numpy as np


class TwoPoisson(WeightingModel):
    def __init__(self, params_file='two_poisson_params.pkl'):
        """
        params_file: path to trained parameters
        """
        with open(params_file, 'rb') as f:
            self.term_params = pickle.load(f)
    
    def scorer(self, searcher, fieldname, text, qf=1):
        return TwoPoissonScorer(searcher, fieldname, text, 
                                 self.term_params, qf)


class TwoPoissonScorer(BaseScorer):
    def __init__(self, searcher, fieldname, text, term_params, qf=1):
        self.searcher = searcher
        self.fieldname = fieldname
        self.text = text
        self.term_params = term_params
        self.qf = qf 
    
    def score(self, matcher):
        """
        Tính score cho document hiện tại
        """
        tf = matcher.weight()
        if tf == 0:
            return 0.0

        if self.text not in self.term_params:
            return 0.0  

        params = self.term_params[self.text]
        lambda_mix = params['lambda']
        mu_elite = params['mu_elite']
        mu_non_elite = params['mu_non_elite']

        lambda_mix = min(max(lambda_mix, 1e-6), 1 - 1e-6)

        log_elite = np.log(lambda_mix) + poisson.logpmf(tf, mu_elite)
        log_non_elite = np.log(1 - lambda_mix) + poisson.logpmf(tf, mu_non_elite)

        return (log_elite - log_non_elite) * self.qf

    def max_quality(self):
        return 100.0

In [103]:
#
def preprocess_query(query):
    """
    Xử lý query GIỐNG NHƯ document preprocessing
    """
    terms = []
    
    segmented = word_segment(query)
    
    for tok in segmented.split():
        cleaned = preprocess(tok)  
        if cleaned:
            terms.append(cleaned)
    
    return " ".join(terms)

In [140]:
#
from whoosh.scoring import TF_IDF, BM25F

def seach_query_with_3_models(idx="ind", raw_query="", top_k=10):
    processed_query = preprocess_query(raw_query)
    print(f"Query gốc: '{raw_query}'")
    print(f"Query đã xử lý: '{processed_query}'")
    
    ix = open_dir(idx)
    qp = QueryParser("content", schema=ix.schema)
    q = qp.parse(processed_query)  
    
    models = {
        'TF-IDF': TF_IDF(),
        'BM25': BM25F(),
        'Two-Poisson': TwoPoisson(params_file='two_poisson_params.pkl')
    }
    
    results_all = {}
    
    for name, model in models.items():
        with ix.searcher(weighting=model) as searcher:
            results = searcher.search(q, limit=top_k)
            docs = [(hit['docid'], hit.score) for hit in results]
            results_all[name] = docs
            
            print(f"\n{name}")
            print("=" * 50)
            if docs:
                for rank, (docid, score) in enumerate(docs, start=1):
                    print(f"{rank}. [{score:.4f}] {docid}")
            else:
                print("Không có kết quả")
    
    ix.close()
    return results_all

In [141]:
seach_query_with_3_models(raw_query="du lịch Hà nội")

Query gốc: 'du lịch Hà nội'
Query đã xử lý: 'du_lịch hà_nội'

TF-IDF
1. [488.0681] Tron_bo_kinh_nghiem_du_lich_vuon_Quoc_gia_Phong_Nha_Ke_Bang
2. [315.5156] Chua_Tam_Chuc_Ha_Nam_-_uoc_menh_danh_la_ngoi_chua_lon_nhat_the_gioi
3. [295.7841] Ban_Ta_Phin_-_Vung_at_Van_Hoa_oc_ao_O_Lao_Cai_-_Du_Lich_Okela
4. [255.9098] Du_lich_Thanh_Co_Quang_Tri_-_Kham_pha_di_san_lich_su_van_hoa_Viet_Nam
5. [194.7019] Tham_quan_khu_di_tich_en_Hung-Phu_Tho_-_AllTours
6. [110.4525] Khu_du_lich_Suoi_Mo_ong_Nai__iem_en_ly_tuong_vao_dip_cuoi_tuan
7. [105.8977] Cam_nang_du_lich_Ha_Noi
8. [88.1795] Kinh_nghiem_tham_quan_chua_Huong_Ha_Noi_chi_tiet_nhat
9. [83.0674] Hoang_thanh_Thang_Long__Wikipedia_tieng_Viet
10. [78.7576] Khu_du_lich_sinh_thai_Lang_Xanh_Ben_Tre

BM25
1. [5.2307] Tron_bo_kinh_nghiem_du_lich_vuon_Quoc_gia_Phong_Nha_Ke_Bang
2. [5.1972] Lang_gom_Bat_Trang_ia_iem_du_lich_trong_ngay_gan_Ha_Noi_-_Vntrip
3. [5.1923] Vuon_Quoc_Gia_Ba_Vi_-_iem_du_lich_sieu_chat_gan_trung_tam_thu_o_-_Vntrip
4. [5.1208] Cam_na

{'TF-IDF': [('Tron_bo_kinh_nghiem_du_lich_vuon_Quoc_gia_Phong_Nha_Ke_Bang',
   488.06809981760864),
  ('Chua_Tam_Chuc_Ha_Nam_-_uoc_menh_danh_la_ngoi_chua_lon_nhat_the_gioi',
   315.5156013610225),
  ('Ban_Ta_Phin_-_Vung_at_Van_Hoa_oc_ao_O_Lao_Cai_-_Du_Lich_Okela',
   295.78407933397443),
  ('Du_lich_Thanh_Co_Quang_Tri_-_Kham_pha_di_san_lich_su_van_hoa_Viet_Nam',
   255.90982105630732),
  ('Tham_quan_khu_di_tich_en_Hung-Phu_Tho_-_AllTours', 194.7018797292918),
  ('Khu_du_lich_Suoi_Mo_ong_Nai__iem_en_ly_tuong_vao_dip_cuoi_tuan',
   110.45248143472828),
  ('Cam_nang_du_lich_Ha_Noi', 105.8976664634891),
  ('Kinh_nghiem_tham_quan_chua_Huong_Ha_Noi_chi_tiet_nhat', 88.17951968231242),
  ('Hoang_thanh_Thang_Long__Wikipedia_tieng_Viet', 83.0674352877397),
  ('Khu_du_lich_sinh_thai_Lang_Xanh_Ben_Tre', 78.75755818837747)],
 'BM25': [('Tron_bo_kinh_nghiem_du_lich_vuon_Quoc_gia_Phong_Nha_Ke_Bang',
   5.230710271684841),
  ('Lang_gom_Bat_Trang_ia_iem_du_lich_trong_ngay_gan_Ha_Noi_-_Vntrip',
   5.197

# Đánh giá mô hình

***2) Xử lý truy vấn***
- Chuẩn bị tập Ground Truth
- Chuẩn bị tập câu truy vấn.
- Truy vấn chỉ mục với mô hình xác suất, trọng số term tính theo TF_IDF

In [125]:
def indexing_cranfield(src, idx="ind_cranfield"):
    import os
    from whoosh.fields import Schema, TEXT, STORED
    from whoosh.analysis import StandardAnalyzer
    from whoosh.index import create_in
    
    if not src.endswith('/'):
        src += '/'

    os.makedirs(idx, exist_ok=True)

    schema = Schema(docid=STORED(), content=TEXT(stored=True, analyzer=StandardAnalyzer()))
    ix = create_in(idx, schema)
    writer = ix.writer()

    files = os.listdir(src)
    for f in files:
        file_path = os.path.join(src, f)
        if not os.path.isfile(file_path):
            continue

        encodings_to_try = ['utf-8', 'utf-16', 'cp1252']
        content = None
        for enc in encodings_to_try:
            try:
                with open(file_path, encoding=enc) as r:
                    content = r.read()
                break
            except UnicodeDecodeError:
                continue

        if content is None:
            print(f"⚠️ Skipped file {f}: could not decode with {encodings_to_try}")
            continue

        try:
            terms = []
            
            for tok in content.split():
                cleaned = preprocess(tok)  
                if cleaned:
                    terms.append(cleaned)

            cont = " ".join(terms)
            writer.add_document(docid=f.split(".")[0], content=cont)

        except Exception as e:
            print(f"Skipped file {f}: {e}")
    
    writer.commit()
    print(f"Indexed {len(files)} documents in {idx}")

In [126]:
data_cranfield = "../4.Indexing/Cranfield/Cranfield"  
indexing_cranfield(data_cranfield, "ind_cranfield") 

Indexed 1400 documents in ind_cranfield


In [127]:
from whoosh.index import open_dir

ix = open_dir("ind_cranfield")
reader = ix.reader()
field = list(reader.schema._fields.keys())[0]
terms = list(reader.field_terms(field))[:1000]
print(f"1000 terms đầu: {terms}")
reader.close()

1000 terms đầu: ['0.000', '0.0001', '0.0005', '0.001', '0.002', '0.003', '0.004', '0.00675', '0.01', '0.010', '0.012', '0.013', '0.014', '0.02', '0.02025', '0.025', '0.028', '0.03', '0.04', '0.05', '0.06', '0.066', '0.08', '0.0904', '0.1', '0.10', '0.11', '0.117', '0.12', '0.120', '0.13', '0.14', '0.1428', '0.14x10', '0.15', '0.15x106', '0.16', '0.18', '0.182', '0.1875', '0.19', '0.195e', '0.199', '0.2', '0.21', '0.211', '0.22', '0.25', '0.254', '0.262', '0.3', '0.30', '0.32', '0.33', '0.35', '0.36', '0.367', '0.368', '0.38', '0.4', '0.40', '0.42', '0.43', '0.44', '0.45', '0.46', '0.5', '0.50', '0.500', '0.53', '0.55', '0.57', '0.5772', '0.58', '0.6', '0.60', '0.62', '0.635', '0.65', '0.69', '0.6x10', '0.7', '0.70', '0.71', '0.715', '0.72', '0.725', '0.73', '0.737', '0.741', '0.75', '0.76', '0.8', '0.80', '0.800', '0.805', '0.823', '0.825', '0.84', '0.840', '0.85', '0.854', '0.8a', '0.9', '0.90', '0.92', '0.95', '0.952', '0.96', '0.98', '000', '000degree', '000degreek', '000k', '0degre

In [128]:
def readGroundTruth(src):
  if src[-1] != '/':
    src += '/'

  GT = {}
  for f in os.listdir(src):
    r = open(src + f)
    rel = {}
    for s in r:
      s = s.strip()
      sp = s.split("\t")
      if len(sp) < 2:
        continue
      did = sp[0].split(" ")[1]
      rel[did] = int(sp[1])
    GT[f.split(".")[0]] = rel
    r.close()
  return GT

In [129]:
GroundTruth = readGroundTruth("../4.Indexing/Cranfield/TEST/RES")
print(GroundTruth)

{'29': {'250': 2, '514': 2, '609': 2, '225': 3, '793': 3, '464': 4, '465': 4, '612': 2, '466': 4, '513': -1}, '15': {'463': 1, '462': 3, '497': -1}, '114': {'919': 1, '916': 2, '920': 3, '921': 3, '895': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '128': {'985': 3, '990': 3, '945': -1}, '129': {'985': 2, '987': 2, '984': 3, '988': 3, '989': 3, '986': 4, '990': 3, '945': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '115': {'51': 2, '185': 2, '878': 3, '874': 4, '184': -1}, '14': {'64': 1, '65': 4, '496': -1}, '28': {'224': 3, '279': 3, '512': -1}, '16': {'266': 2, '106': 3, '196': 3, '498': -1}, '103': {'826': 3, '828': 3, '761': -1}, '117': {'360': 2, '605': 3, '896': -1}, '116': {'922': 1, '360': 3, '605': 3, '927': 3, '492': 4, '896': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '17': {'106': 2, '196': 3, '498': -1}, '13': {'64': 2, '265': 2, '65': 4, 

In [130]:
def readQuery(src):
    qry = {}
    with open(src, 'r', encoding='utf-8') as r:
        for s in r:
            s = s.strip()
            ps = s.split("\t")
            
            terms = []

            for tok in ps[1].split():
                tok = preprocess(tok)
                if tok != None:
                    terms.append(tok)
                    
            qry[ps[0]] = " ".join(terms)
    
    return qry

In [131]:
Queries = readQuery("../4.Indexing/Cranfield/TEST/query.txt")
print(Queries)

{'1': 'what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft', '2': 'what are the structural and aeroelastic problems associated with flight of high speed aircraft', '3': 'what problems of heat conduction in composite slabs have been solved far', '4': 'can a criterion be developed to show empirically the validity of flow solutions for chemically reacting gas mixtures based on the simplifying assumption of instantaneous local chemical equilibrium', '5': 'what chemical kinetic system is applicable to hypersonic aerodynamic problems', '6': 'what theoretical and experimental guides we have as to turbulent couette flow behaviour', '7': 'is it possible to relate the available pressure distributions for an ogive forebody at zero angle of attack to the lower surface pressures of an equivalent ogive forebody at angle of attack', '8': 'what methods -dash exact or approximate -dash are presently available for predicting body pressures at angle of a

In [142]:
# bỏ qua không quan tâm
def processQueries(ind, qry):

  idx = index.open_dir(ind)
  searcher = idx.searcher(weighting=scoring.BM25F())
  parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)

  RET = {}
  for key in qry:
    query = parser.parse(qry[key])
    results = searcher.search(query, limit=None)
    rel = {}
    for i in range(len(results)):
      rel[results[i]["docid"]] = results[i].score
    RET[key] = rel
  return RET

In [ ]:
#RunResults = processQueries("ind_cranfield", Queries)

In [143]:
def processQueries_3models(ind, qry):
    """
    Xử lý queries với 3 models: TF-IDF, BM25, Two-Poisson
    """
    from whoosh.scoring import TF_IDF, BM25F
    
    idx = index.open_dir(ind)
    parser = qparser.QueryParser("content", idx.schema, group=qparser.OrGroup)
    
    models = {
        'tfidf': TF_IDF(),
        'bm25': BM25F(),
        'two_poisson': TwoPoisson(params_file='two_poisson_params.pkl')
    }
    
    results_all_models = {}
    
    for model_name, weighting in models.items():
        print(f"Processing queries with {model_name}...")
        searcher = idx.searcher(weighting=weighting)
        
        RET = {}
        for key in qry:
            try:
                query = parser.parse(qry[key])
                results = searcher.search(query, limit=None)
                
                rel = {}
                for i in range(len(results)):
                    rel[results[i]["docid"]] = results[i].score
                RET[key] = rel
                
            except Exception as e:
                print(f"Error processing query {key}: {e}")
                RET[key] = {}
        
        results_all_models[model_name] = RET
        searcher.close()
    
    idx.close()
    return results_all_models


In [144]:
def calculate_precision_at_k(retrieved, relevant, k):
    """
    Tính Precision@K
    """
    if k == 0 or len(retrieved) == 0:
        return 0.0
    
    retrieved_at_k = retrieved[:k]
    relevant_count = sum(1 for doc in retrieved_at_k if doc in relevant)
    
    return relevant_count / k


def calculate_recall_at_k(retrieved, relevant, k):
    """
    Tính Recall@K
    """
    if len(relevant) == 0:
        return 0.0
    
    retrieved_at_k = retrieved[:k]
    relevant_count = sum(1 for doc in retrieved_at_k if doc in relevant)
    
    return relevant_count / len(relevant)


def calculate_average_precision(retrieved, relevant):
    """
    Tính Average Precision (AP)
    """
    if len(relevant) == 0:
        return 0.0
    
    relevant_count = 0
    sum_precisions = 0.0
    
    for i, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            relevant_count += 1
            precision_at_i = relevant_count / i
            sum_precisions += precision_at_i
    
    if relevant_count == 0:
        return 0.0
    
    return sum_precisions / len(relevant)


def evaluate_model(run_results, ground_truth):
    metrics = {
        'MAP': 0.0,
        'P@5': 0.0,
        'P@10': 0.0,
        'R@5': 0.0,
        'R@10': 0.0,
    }
    
    num_queries = 0
    
    for query_id in ground_truth.keys():
        if query_id not in run_results:
            continue
        
        relevant_docs = {doc: rel for doc, rel in ground_truth[query_id].items() 
                        if rel > 0}
        
        if len(relevant_docs) == 0: 
            continue
        
        retrieved_docs = sorted(run_results[query_id].items(), 
                               key=lambda x: x[1], 
                               reverse=True) 
        retrieved_list = [doc for doc, score in retrieved_docs]
        
        ap = calculate_average_precision(retrieved_list, relevant_docs.keys())
        p5 = calculate_precision_at_k(retrieved_list, relevant_docs.keys(), 5)
        p10 = calculate_precision_at_k(retrieved_list, relevant_docs.keys(), 10)
        r5 = calculate_recall_at_k(retrieved_list, relevant_docs.keys(), 5)
        r10 = calculate_recall_at_k(retrieved_list, relevant_docs.keys(), 10)
        
        metrics['MAP'] += ap
        metrics['P@5'] += p5
        metrics['P@10'] += p10
        metrics['R@5'] += r5
        metrics['R@10'] += r10
        
        num_queries += 1
    
    if num_queries > 0:
        for key in metrics:
            metrics[key] /= num_queries
    
    return metrics, num_queries



In [145]:
def compare_all_models(results_all_models, ground_truth):
    """
    So sánh 3 models
    """
    print("\n" + "="*70)
    print("EVALUATION RESULTS - COMPARISON OF 3 MODELS")
    print("="*70)
    
    comparison = {}
    
    for model_name, run_results in results_all_models.items():
        print(f"\nEvaluating {model_name.upper()}...")
        metrics, num_queries = evaluate_model(run_results, ground_truth)
        comparison[model_name] = metrics
        
        print(f"   Evaluated on {num_queries} queries")
        print(f"   MAP:      {metrics['MAP']:.4f}")
        print(f"   P@5:      {metrics['P@5']:.4f}")
        print(f"   P@10:     {metrics['P@10']:.4f}")
        print(f"   R@5:      {metrics['R@5']:.4f}")
        print(f"   R@10:     {metrics['R@10']:.4f}")
    
    print("\n" + "="*70)
    print("COMPARISON TABLE")
    print("="*70)
    print(f"{'Metric':<15} {'TF-IDF':<15} {'BM25':<15} {'Two-Poisson':<15}")
    print("-"*70)
    
    for metric in ['MAP', 'P@5', 'P@10', 'R@5', 'R@10']:
        tfidf_val = comparison['tfidf'][metric]
        bm25_val = comparison['bm25'][metric]
        tp_val = comparison['two_poisson'][metric]
        
        print(f"{metric:<15} {tfidf_val:<15.4f} {bm25_val:<15.4f} {tp_val:<15.4f}")
    
    # Tìm model tốt nhất cho từng metric 
    print("\n" + "="*70)
    print("BEST MODEL FOR EACH METRIC")
    print("="*70)
    
    for metric in ['MAP', 'P@5', 'P@10', 'R@5', 'R@10']:
        values = {model: comparison[model][metric] for model in comparison}
        best_model = max(values, key=values.get)
        best_value = values[best_model]
        
        print(f"{metric:<15} → {best_model.upper():<15} ({best_value:.4f})")
    
    return comparison


# Process queries và truy vấn với 3 models
results_all = processQueries_3models("ind_cranfield", Queries)

# Đánh giá và so sánh với 3 model
comparison_results = compare_all_models(results_all, GroundTruth)


Processing queries with tfidf...
Processing queries with bm25...
Processing queries with two_poisson...

EVALUATION RESULTS - COMPARISON OF 3 MODELS

Evaluating TFIDF...
   Evaluated on 225 queries
   MAP:      0.1987
   P@5:      0.2000
   P@10:     0.1569
   R@5:      0.1760
   R@10:     0.2607

Evaluating BM25...
   Evaluated on 225 queries
   MAP:      0.2694
   P@5:      0.2978
   P@10:     0.2173
   R@5:      0.2689
   R@10:     0.3690

Evaluating TWO_POISSON...
   Evaluated on 225 queries
   MAP:      0.0316
   P@5:      0.0329
   P@10:     0.0280
   R@5:      0.0303
   R@10:     0.0475

COMPARISON TABLE
Metric          TF-IDF          BM25            Two-Poisson    
----------------------------------------------------------------------
MAP             0.1987          0.2694          0.0316         
P@5             0.2000          0.2978          0.0329         
P@10            0.1569          0.2173          0.0280         
R@5             0.1760          0.2689          0.0303

Đánh giá trên mẫu thử